# P5-Real: WILDS External Validation
**UAI 2026 Rebuttal - Paper #43**

Run this notebook in Google Colab to validate SHAP concentration diagnostic on real WILDS benchmarks.

In [ ]:
# Install dependencies
!pip install -q wilds lightgbm shap scikit-learn

In [ ]:
import json
import warnings
import numpy as np
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

import lightgbm as lgb
import shap
from wilds import get_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_covtype, fetch_openml
from sklearn.preprocessing import LabelEncoder

In [ ]:
def run_shap_diagnostic(X_train, y_train, X_val, y_val, X_test, y_test, dataset_name):
    """Run SHAP concentration diagnostic and conformal prediction."""
    print(f"\n{'='*50}")
    print(f"Running diagnostic on: {dataset_name}")
    print(f"{'='*50}")
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    n_classes = len(np.unique(y_train))
    if n_classes < 2:
        print(f"Skipping {dataset_name}: only {n_classes} class(es)")
        return None

    # Train LightGBM
    model = lgb.LGBMClassifier(
        n_estimators=100,
        num_leaves=31,
        learning_rate=0.05,
        verbose=-1,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # SHAP concentration
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val[:min(500, len(X_val))])

    if isinstance(shap_values, list):
        shap_importance = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        shap_importance = np.abs(shap_values).mean(axis=0)

    total_importance = shap_importance.sum()
    top1_importance = shap_importance.max()
    concentration = (top1_importance / total_importance * 100) if total_importance > 0 else 0

    print(f"SHAP concentration (top-1): {concentration:.1f}%")

    # Conformal prediction (APS)
    val_probs = model.predict_proba(X_val)
    test_probs = model.predict_proba(X_test)

    def compute_aps_scores(probs, y_true):
        scores = []
        for i in range(len(y_true)):
            sorted_idx = np.argsort(-probs[i])
            cumsum = 0
            for idx in sorted_idx:
                cumsum += probs[i, idx]
                if idx == y_true[i]:
                    scores.append(cumsum - probs[i, idx] * np.random.rand())
                    break
            else:
                scores.append(1.0)
        return np.array(scores)

    val_scores = compute_aps_scores(val_probs, y_val)
    test_scores = compute_aps_scores(test_probs, y_test)

    alpha = 0.1
    q_hat = np.quantile(val_scores, 1 - alpha)

    val_coverage = np.mean(val_scores <= q_hat)
    test_coverage = np.mean(test_scores <= q_hat)
    coverage_drop = val_coverage - test_coverage

    print(f"Val coverage: {val_coverage:.3f}")
    print(f"Test coverage: {test_coverage:.3f}")
    print(f"Coverage drop: {coverage_drop*100:.1f}%")

    if coverage_drop > 0.5:
        category = 'Catastrophic'
    elif coverage_drop > 0.15:
        category = 'Severe'
    else:
        category = 'Robust'

    print(f"Category: {category}")

    return {
        'dataset': dataset_name,
        'concentration': float(concentration),
        'val_coverage': float(val_coverage),
        'test_coverage': float(test_coverage),
        'coverage_drop': float(coverage_drop),
        'coverage_drop_pct': float(coverage_drop * 100),
        'category': category,
        'n_train': X_train.shape[0],
        'n_features': X_train.shape[1],
        'n_classes': int(n_classes),
    }

## 1. WILDS CivilComments (Demographic Shift)

In [ ]:
print("Loading WILDS CivilComments...")
dataset = get_dataset(dataset='civilcomments', download=True)

train_data = dataset.get_subset('train')
val_data = dataset.get_subset('val')
test_data = dataset.get_subset('test')

n_train, n_val, n_test = 10000, 3000, 3000
print(f"Sampling: train={n_train}, val={n_val}, test={n_test}")

train_texts = [str(train_data[i][0]) for i in range(n_train)]
val_texts = [str(val_data[i][0]) for i in range(n_val)]
test_texts = [str(test_data[i][0]) for i in range(n_test)]

y_train = np.array([int(train_data[i][1]) for i in range(n_train)])
y_val = np.array([int(val_data[i][1]) for i in range(n_val)])
y_test = np.array([int(test_data[i][1]) for i in range(n_test)])

print("Extracting TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
X_train_cc = vectorizer.fit_transform(train_texts).toarray()
X_val_cc = vectorizer.transform(val_texts).toarray()
X_test_cc = vectorizer.transform(test_texts).toarray()

result_cc = run_shap_diagnostic(X_train_cc, y_train, X_val_cc, y_val, X_test_cc, y_test, 'civilcomments')
result_cc['source'] = 'wilds'
result_cc['shift_type'] = 'demographic'

## 2. WILDS Amazon (User/Time Shift)

In [ ]:
print("Loading WILDS Amazon...")
dataset = get_dataset(dataset='amazon', download=True)

train_data = dataset.get_subset('train')
val_data = dataset.get_subset('id_val')
test_data = dataset.get_subset('test')

n_train, n_val, n_test = 10000, 3000, 3000
print(f"Sampling: train={n_train}, val={n_val}, test={n_test}")

train_texts = [str(train_data[i][0]) for i in range(n_train)]
val_texts = [str(val_data[i][0]) for i in range(n_val)]
test_texts = [str(test_data[i][0]) for i in range(n_test)]

y_train = np.array([int(train_data[i][1]) for i in range(n_train)])
y_val = np.array([int(val_data[i][1]) for i in range(n_val)])
y_test = np.array([int(test_data[i][1]) for i in range(n_test)])

print("Extracting TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
X_train_am = vectorizer.fit_transform(train_texts).toarray()
X_val_am = vectorizer.transform(val_texts).toarray()
X_test_am = vectorizer.transform(test_texts).toarray()

result_am = run_shap_diagnostic(X_train_am, y_train, X_val_am, y_val, X_test_am, y_test, 'amazon')
result_am['source'] = 'wilds'
result_am['shift_type'] = 'user_time'

## 3. WILDS FMoW (Temporal Shift)

In [ ]:
# FMoW requires image processing - skip if resources limited
# Uses satellite imagery with temporal shift
try:
    print("Loading WILDS FMoW (satellite imagery)...")
    dataset = get_dataset(dataset='fmow', download=True)
    
    # Extract simple pixel statistics as features
    def extract_image_features(subset, n_samples=3000):
        features = []
        labels = []
        for i in range(min(n_samples, len(subset))):
            img, y, _ = subset[i]
            img = np.array(img)
            # Simple features: mean, std per channel, overall stats
            feat = [
                img.mean(), img.std(),
                img[:,:,0].mean() if len(img.shape)==3 else img.mean(),
                img[:,:,1].mean() if len(img.shape)==3 else img.mean(),
                img[:,:,2].mean() if len(img.shape)==3 else img.mean(),
            ]
            features.append(feat)
            labels.append(int(y))
        return np.array(features), np.array(labels)
    
    X_train_fm, y_train_fm = extract_image_features(dataset.get_subset('train'))
    X_val_fm, y_val_fm = extract_image_features(dataset.get_subset('id_val'))
    X_test_fm, y_test_fm = extract_image_features(dataset.get_subset('test'))
    
    result_fm = run_shap_diagnostic(X_train_fm, y_train_fm, X_val_fm, y_val_fm, X_test_fm, y_test_fm, 'fmow')
    result_fm['source'] = 'wilds'
    result_fm['shift_type'] = 'temporal'
except Exception as e:
    print(f"FMoW failed (expected if no GPU): {e}")
    result_fm = None

## 4. Covertype (Temporal/Regional Shift)

In [ ]:
print("Loading Covertype...")
data = fetch_covtype()
X, y = data.data, data.target

n = len(X)
X_train_ct, y_train_ct = X[:int(0.7*n)], y[:int(0.7*n)]
X_val_ct, y_val_ct = X[int(0.7*n):int(0.85*n)], y[int(0.7*n):int(0.85*n)]
X_test_ct, y_test_ct = X[int(0.85*n):], y[int(0.85*n):]

result_ct = run_shap_diagnostic(X_train_ct, y_train_ct, X_val_ct, y_val_ct, X_test_ct, y_test_ct, 'covertype')
result_ct['source'] = 'sklearn'
result_ct['shift_type'] = 'temporal_region'

## 5. Adult Income (Age Demographic Shift)

In [ ]:
print("Loading Adult Income...")
data = fetch_openml('adult', version=2, as_frame=True)
df = data.data
y = (data.target == '>50K').astype(int).values

# Encode categorical
X_encoded = []
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        le = LabelEncoder()
        X_encoded.append(le.fit_transform(df[col].astype(str)))
    else:
        X_encoded.append(df[col].fillna(0).values)
X = np.column_stack(X_encoded)

# Age-based shift
age_col = df.columns.tolist().index('age')
ages = X[:, age_col]
young_mask = ages < 40
old_mask = ages >= 40

X_young, y_young = X[young_mask], y[young_mask]
X_old, y_old = X[old_mask], y[old_mask]

n_young = len(X_young)
X_train_ad, y_train_ad = X_young[:int(0.7*n_young)], y_young[:int(0.7*n_young)]
X_val_ad, y_val_ad = X_young[int(0.7*n_young):], y_young[int(0.7*n_young):]
X_test_ad, y_test_ad = X_old[:3000], y_old[:3000]

result_ad = run_shap_diagnostic(X_train_ad, y_train_ad, X_val_ad, y_val_ad, X_test_ad, y_test_ad, 'adult_age_shift')
result_ad['source'] = 'openml'
result_ad['shift_type'] = 'demographic_age'

## Summary & Results

In [ ]:
# Collect all results
all_results = [r for r in [result_cc, result_am, result_fm, result_ct, result_ad] if r is not None]

print("\n" + "="*70)
print("SUMMARY: Real Benchmark Validation")
print("="*70)

print(f"\n{'Dataset':<20} {'Source':<10} {'Shift':<15} {'Conc':>8} {'Drop':>10} {'Category':<12}")
print("-" * 80)

for r in all_results:
    print(f"{r['dataset']:<20} {r.get('source', 'N/A'):<10} {r.get('shift_type', 'N/A'):<15} "
          f"{r['concentration']:>8.1f}% {r['coverage_drop_pct']:>+10.1f}% {r['category']:<12}")

# Correlation
concentrations = [r['concentration'] for r in all_results]
drops = [r['coverage_drop_pct'] for r in all_results]

rho, p = spearmanr(concentrations, drops)
print(f"\nSpearman correlation: ρ = {rho:.3f} (p = {p:.4f})")

# Threshold accuracy
threshold = 40
predictions = [r['concentration'] > threshold for r in all_results]
actuals = [r['coverage_drop_pct'] > 15 for r in all_results]
accuracy = sum(p == a for p, a in zip(predictions, actuals)) / len(all_results)
print(f"Threshold (40%) accuracy: {accuracy*100:.0f}%")

# Group stats
catastrophic = [r for r in all_results if r['category'] == 'Catastrophic']
severe = [r for r in all_results if r['category'] == 'Severe']
robust = [r for r in all_results if r['category'] == 'Robust']

print(f"\nCatastrophic (n={len(catastrophic)}): mean C = {np.mean([r['concentration'] for r in catastrophic]):.1f}%" if catastrophic else "")
print(f"Severe (n={len(severe)}): mean C = {np.mean([r['concentration'] for r in severe]):.1f}%" if severe else "")
print(f"Robust (n={len(robust)}): mean C = {np.mean([r['concentration'] for r in robust]):.1f}%" if robust else "")

In [ ]:
# Save results as JSON (copy this output)
output = {
    'results': all_results,
    'summary': {
        'n_datasets': len(all_results),
        'n_wilds': len([r for r in all_results if r.get('source') == 'wilds']),
        'spearman_rho': float(rho),
        'spearman_p': float(p),
        'threshold_accuracy': float(accuracy),
    },
    'methodology': 'WILDS (CivilComments, Amazon, FMoW) + sklearn (Covertype) + OpenML (Adult)'
}

print("\n" + "="*70)
print("COPY THIS JSON FOR REBUTTAL:")
print("="*70)
print(json.dumps(output, indent=2))